## Tutorial for CellGRN and CellGRN-sparse
- Datasets can be downloaded from Figshare repository from 10.6084/m9.figshare.31304758. Please uncompress the tar.gz file.
- Please install cellgrn package and activating ipykernel before running this tutorial.

In [ ]:
# 1. load required packages
from cellgrn.main import normalzie_rna,parse_edges,compute_all_cells_grn,summarize_grn,format_sample_grn,format_celltype_grn,  SparseGRNCalculator, normalzie_rna_sparse

import numpy as np
import pandas as pd
import os
import anndata as ad
from scipy import sparse
import gc
import pickle

In [ ]:
# 2. we use linger grn result as example backbone
cand_df = pd.read_csv(f"/home/shaliu_fu/multireg/cellGRN_data/scalability/linger_grn.csv",header=0)

input_genes = [i.rstrip() for i in open(f"/home/shaliu_fu/multireg/cellGRN_data/scalability/linger_gene.txt")]
input_peaks = [i.rstrip() for i in open(f"/home/shaliu_fu/multireg/cellGRN_data/scalability/linger_peak.txt")]
all_tf = [i.rstrip() for i in open("/home/shaliu_fu/multireg/cellGRN_code/all_hg_TF.txt")] 

In [8]:
# 3. load a 500-cells multiome dataset 

base_path = f"/home/shaliu_fu/multireg/benchmark/bench_dataset/scalability/c500/"
        
# 1. 加载数据
cell_meta = pd.read_csv(f"{base_path}/metadata.csv", index_col=0)
cell_types = cell_meta['cell_type.l1'].values

input_rna = ad.read_h5ad(f"{base_path}/BMMC-multiome-c500-RNA-counts.h5ad")
input_atac = ad.read_h5ad(f"{base_path}/BMMC-multiome-c500-ATAC-peaks.h5ad")

input_gene = input_rna.var.index.values
input_peak = input_atac.var.index.values
input_tf = list(set(input_gene) & set(all_tf))
cell_types = cell_meta['cell_type.l1']


input_df1 = pd.DataFrame(input_rna.X.toarray(),index=input_rna.obs.index.values,columns=input_rna.var.index.values)
peak_rename = [i.replace("-",":",1) for i in input_atac.var.index.values]
input_df2 = pd.DataFrame(input_atac.X.toarray(),index=input_atac.obs.index.values,columns=peak_rename)

# input_peaks = [i.replace(":","-") for i in input_peaks]
input_df1 = input_df1[input_genes]
input_df2 = input_df2[input_peaks]


rna_data1,rna_data2 = normalzie_rna(input_df1)
atac_data = input_df2.copy()

input_tfs = [tf for tf in input_tf if tf in input_genes]
tf_data1 = rna_data1[input_tfs].copy()
tf_data2 = rna_data2[input_tfs].copy()

In [ ]:
# 4.run with cellGRN
edges_idx,edges_name = parse_edges(cand_df, input_tfs, input_genes, input_peaks)

grn_scale2 = compute_all_cells_grn(tf_data2, rna_data2, atac_data,edges_idx, edges_name,
    input_tfs, input_genes, input_peaks)

sample_grn_scale2, celltype_grn_scale2 = summarize_grn(grn_scale2, cell_types)


tf_gene_res_scale2, tf_peak_res_scale2, gene_peak_res_scale2 = format_sample_grn(sample_grn_scale2)
tf_gene_ct_res_scale2, tf_peak_ct_res_scale2, gene_peak_ct_res_scale2 = format_celltype_grn(celltype_grn_scale2)

In [ ]:
# 5.save cell-specific, cell type and sample-wise GRN results.
# cell specific matrix as pkl file.
outdir = "./output/"
with open(f"{outdir}/demo_linger_cell_grn.pkl", "wb") as f:
    pickle.dump(grn_scale2.copy(), f)

tf_gene_res_scale2.to_csv(os.path.join(outdir, "tf_gene_sample_scale2.csv"), index=False)
tf_peak_res_scale2.to_csv(os.path.join(outdir, "tf_peak_sample_scale2.csv"), index=False)
gene_peak_res_scale2.to_csv(os.path.join(outdir, "gene_peak_sample_scale2.csv"), index=False)

tf_gene_ct_res_scale2.to_csv(os.path.join(outdir, "tf_gene_celltype_scale2.csv"), index=False)
tf_peak_ct_res_scale2.to_csv(os.path.join(outdir, "tf_peak_celltype_scale2.csv"), index=False)
gene_peak_ct_res_scale2.to_csv(os.path.join(outdir, "gene_peak_celltype_scale2.csv"), index=False)

In [ ]:
# (optional) run with CellGRN-sparse
input_rna = ad.read_h5ad(f"{base_path}/BMMC-multiome-c500-RNA-counts.h5ad")
input_atac = ad.read_h5ad(f"{base_path}/BMMC-multiome-c500-ATAC-peaks.h5ad")
cell_types = cell_meta['cell_type.l1'].values

peak_rename = [i.replace("-",":",1) for i in input_atac.var.index.values]
input_atac.var.index = peak_rename

gene_indices = [input_rna.var_names.get_loc(g) for g in input_genes if g in input_rna.var_names]
peak_indices = [input_atac.var_names.get_loc(p) for p in input_peaks if p in input_atac.var_names]

rna_sub = input_rna[:, gene_indices].copy()
atac_sub = input_atac[:, peak_indices].copy()

current_genes = rna_sub.var.index.values
current_peaks = atac_sub.var.index.values

input_tfs = [tf for tf in current_genes if tf in all_tf]
tf_indices_in_sub = [rna_sub.var_names.get_loc(tf) for tf in input_tfs]


rna_data1_sparse, rna_data2_dense = normalzie_rna_sparse(rna_sub.X)
tf_data2_arr = rna_data2_dense[:, tf_indices_in_sub]


edges_idx, edges_name = parse_edges(cand_df, input_tfs, current_genes, current_peaks)
calculator = SparseGRNCalculator(
    edges_idx, edges_name,
    input_tfs, current_genes, current_peaks
)


batch_size = 1000 
n_total = rna_sub.shape[0]
atac_sparse = atac_sub.X 

for i in range(0, n_total, batch_size):
    end_i = min(i + batch_size, n_total)
    
    batch_tf = tf_data2_arr[i:end_i]
    batch_rna = rna_data2_dense[i:end_i]
    batch_atac = atac_sparse[i:end_i]
    batch_ct = cell_types[i:end_i]
    
    calculator.process_batch(batch_tf, batch_rna, batch_atac, batch_ct)
    
    if i % 5000 == 0:
        gc.collect() 

sample_grn_scale2, celltype_grn_scale2 = calculator.finalize()

# Note: CellGRN-sparse DO NOT output cell-specific results.

In [22]:
cell_types = cell_meta['cell_type.l1'].values

In [ ]:
batch_ct = cell_types[i:end_i]